In [1]:
# ============================================================================
# Drought characterization — Rio Parana basin
# Method: Run Theory (Yevjevich, 1967)
# Index: SSI-12
# Metrics: Frequency, Duration (median), Severity (median)
# Definition: event starts when SSI < -1 for >= 2 consecutive months
#             event ends when SSI >= 0
# Reference: Spinoni et al. (2014), Tabari et al. (2021)
# Scenarios: Historical (1985-2014), SSP126 (2041-2070), SSP585 (2041-2070)
# Models: MPI-ESM1-2-HR, UKESM1-0-LL, GFDL-ESM4, IPSL-CM6A-LR, MRI-ESM2-0
# ============================================================================

import os
import numpy as np
import xarray as xr
import netCDF4 as nc
from pathlib import Path

# --- CONFIGURATION ---
models    = ["MPI-ESM1-2-HR", "UKESM1-0-LL", "GFDL-ESM4", "IPSL-CM6A-LR", "MRI-ESM2-0"]
scenarios = ["historical", "ssp126", "ssp585"]
basin     = "parana"
SSI_VAR   = "SSI_12"

path_in  = "/data/brussel/vo/000/bvo00033/vsc11346/SSI_R/3.Outputs_SSI/2.baseline_105/Rio_Parana/"
path_out = "/data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/"


# ============================================================================
# FUNCTION: Calculate drought metrics for a single time series
# ============================================================================
def calc_drought_metrics(ssi_ts):
    """
    Calculates drought frequency, duration and severity using run theory.

    Event definition:
        Start: SSI < -1 for >= 2 consecutive months
        End:   SSI >= 0
    """
    n           = len(ssi_ts)
    in_event    = False
    consec_neg  = 0
    event_start = None
    durations   = []
    severities  = []

    for t in range(n):
        val = ssi_ts[t]

        if np.isnan(val):
            in_event   = False
            consec_neg = 0
            continue

        if not in_event:
            if val < -1:
                consec_neg += 1
                if consec_neg >= 2:
                    in_event    = True
                    event_start = t - 1
            else:
                consec_neg = 0
        else:
            if val >= 0:
                duration = (t - 1) - event_start + 1
                severity = np.sum(np.abs(ssi_ts[event_start:t]))
                durations.append(duration)
                severities.append(severity)
                in_event   = False
                consec_neg = 0

    if in_event:
        duration = (n - 1) - event_start + 1
        severity = np.sum(np.abs(ssi_ts[event_start:n]))
        durations.append(duration)
        severities.append(severity)

    frequency = len(durations)

    if frequency == 0:
        return frequency, np.nan, np.nan

    return frequency, np.median(durations), np.median(severities)


# ============================================================================
# FUNCTION: Save drought metrics as NetCDF
# ============================================================================
def save_netcdf(freq_grid, dur_grid, sev_grid, lon, lat,
                basin, scenario, model, output_dir):
    """
    Saves drought metrics (frequency, duration, severity) into a single NetCDF.
    Arrays are stored in (lat, lon) order following CF conventions.
    """
    output_file = f"{output_dir}drought_metrics_{basin}_{scenario}_{model}.nc"
    ds_out      = nc.Dataset(output_file, "w", format="NETCDF4")

    ds_out.createDimension("lat", len(lat))
    ds_out.createDimension("lon", len(lon))

    lat_var       = ds_out.createVariable("lat", "f4", ("lat",))
    lat_var.units = "degrees_north"
    lat_var[:]    = lat

    lon_var       = ds_out.createVariable("lon", "f4", ("lon",))
    lon_var.units = "degrees_east"
    lon_var[:]    = lon

    freq_save = np.where(np.isnan(freq_grid), -9999, freq_grid)
    dur_save  = np.where(np.isnan(dur_grid),  -9999, dur_grid)
    sev_save  = np.where(np.isnan(sev_grid),  -9999, sev_grid)

    freq_var           = ds_out.createVariable("frequency", "f4", ("lat", "lon"),
                                                fill_value=-9999)
    freq_var.units     = "count"
    freq_var.long_name = "Number of drought events"
    freq_var[:]        = freq_save

    dur_var           = ds_out.createVariable("duration", "f4", ("lat", "lon"),
                                               fill_value=-9999)
    dur_var.units     = "months"
    dur_var.long_name = "Median drought duration"
    dur_var[:]        = dur_save

    sev_var           = ds_out.createVariable("severity", "f4", ("lat", "lon"),
                                               fill_value=-9999)
    sev_var.units     = "-"
    sev_var.long_name = "Median drought severity (sum of |SSI| per event)"
    sev_var[:]        = sev_save

    ds_out.description = (f"Drought characteristics — {basin} basin, "
                          f"{scenario}, {model}. "
                          f"Run theory (Yevjevich 1967): "
                          f"start SSI < -1 for >= 2 months, end SSI >= 0.")
    ds_out.index       = "SSI-12"
    ds_out.basin       = basin
    ds_out.scenario    = scenario
    ds_out.model       = model

    ds_out.close()
    print(f"    Saved: {output_file}")


# ============================================================================
# FUNCTION: Calculate ensemble median across 5 models
# ============================================================================
def calc_ensemble(basin, scenario, models, output_dir):
    """
    Reads individual model NetCDFs and calculates pixel-wise median
    across all models for each metric.
    """
    freq_stack = []
    dur_stack  = []
    sev_stack  = []

    for model in models:
        nc_file = f"{output_dir}drought_metrics_{basin}_{scenario}_{model}.nc"
        ds      = xr.open_dataset(nc_file)

        freq = ds["frequency"].values.astype(float)
        dur  = ds["duration"].values.astype(float)
        sev  = ds["severity"].values.astype(float)

        freq = np.where(freq == -9999, np.nan, freq)
        dur  = np.where(dur  == -9999, np.nan, dur)
        sev  = np.where(sev  == -9999, np.nan, sev)

        freq_stack.append(freq)
        dur_stack.append(dur)
        sev_stack.append(sev)

        lon = ds["lon"].values
        lat = ds["lat"].values
        ds.close()

    freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
    dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
    sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)

    save_netcdf(freq_ensemble, dur_ensemble, sev_ensemble,
                lon, lat, basin, scenario, "ensemble", output_dir)

    print(f"\n  Ensemble summary — {scenario}:")
    for name, grid in [("FREQUENCY", freq_ensemble),
                       ("DURATION",  dur_ensemble),
                       ("SEVERITY",  sev_ensemble)]:
        valid = grid[~np.isnan(grid)]
        print(f"    {name}: min={valid.min():.2f}, "
              f"max={valid.max():.2f}, "
              f"median={np.median(valid):.2f}")


# ============================================================================
# MAIN LOOP — SCENARIOS x MODELS
# ============================================================================
for scenario in scenarios:

    out_dir = f"{path_out}{scenario}/"
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"SCENARIO: {scenario.upper()} | BASIN: {basin.upper()}")
    print(f"{'='*60}")

    for model in models:

        print(f"\n  Processing: {model}")

        # Read SSI-12 NetCDF
        nc_file = f"{path_in}{scenario}/ssi12_{basin}_{scenario}_{model}.nc"
        ds      = xr.open_dataset(nc_file)
        ssi     = ds[SSI_VAR].values
        lon     = ds["lon"].values
        lat     = ds["lat"].values
        dims    = ds[SSI_VAR].dims
        ds.close()

        # Replace fill values
        ssi = np.where(ssi == -9999, np.nan, ssi)

        # Ensure (lat, lon, time) order — input comes as (time, lat, lon)
        if dims[0] == "time":
            ssi = np.transpose(ssi, (1, 2, 0))

        n_lat, n_lon, _ = ssi.shape

        # Initialize output grids — shape (n_lat, n_lon)
        freq_grid = np.full((n_lat, n_lon), np.nan)
        dur_grid  = np.full((n_lat, n_lon), np.nan)
        sev_grid  = np.full((n_lat, n_lon), np.nan)

        # Pixel-by-pixel calculation
        for i in range(n_lat):
            for j in range(n_lon):
                ts = ssi[i, j, :]
                if np.all(np.isnan(ts)):
                    continue
                (freq_grid[i, j],
                 dur_grid[i, j],
                 sev_grid[i, j]) = calc_drought_metrics(ts)

        # Print model summary
        for name, grid in [("FREQUENCY", freq_grid),
                           ("DURATION",  dur_grid),
                           ("SEVERITY",  sev_grid)]:
            valid = grid[~np.isnan(grid)]
            print(f"    {name}: min={valid.min():.2f}, "
                  f"max={valid.max():.2f}, "
                  f"median={np.median(valid):.2f}")

        # Save individual model NetCDF
        save_netcdf(freq_grid, dur_grid, sev_grid,
                    lon, lat, basin, scenario, model, out_dir)

    # Calculate and save ensemble median
    print(f"\n  Computing ensemble median — {scenario}")
    calc_ensemble(basin, scenario, models, out_dir)


print(f"\n{'='*60}")
print(f"COMPLETED — Rio Parana basin, 5 models, 3 scenarios")
print(f"Output: {path_out}")
print(f"{'='*60}")


SCENARIO: HISTORICAL | BASIN: PARANA

  Processing: MPI-ESM1-2-HR
    FREQUENCY: min=0.00, max=10.00, median=5.00
    DURATION: min=3.00, max=70.00, median=16.50
    SEVERITY: min=2.54, max=67.47, median=16.06
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/historical/drought_metrics_parana_historical_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=0.00, max=9.00, median=5.00
    DURATION: min=4.00, max=115.00, median=15.00
    SEVERITY: min=5.28, max=79.75, median=16.81
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/historical/drought_metrics_parana_historical_UKESM1-0-LL.nc

  Processing: GFDL-ESM4
    FREQUENCY: min=0.00, max=10.00, median=5.00
    DURATION: min=2.00, max=58.00, median=16.00
    SEVERITY: min=2.48, max=59.71, median=18.15
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/historical/drou

/tmp/ipykernel_2337151/3510858072.py:176: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_2337151/3510858072.py:177: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_2337151/3510858072.py:178: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/historical/drought_metrics_parana_historical_ensemble.nc

  Ensemble summary — historical:
    FREQUENCY: min=0.00, max=8.00, median=5.00
    DURATION: min=2.00, max=74.00, median=16.00
    SEVERITY: min=2.48, max=42.83, median=16.38

SCENARIO: SSP126 | BASIN: PARANA

  Processing: MPI-ESM1-2-HR
    FREQUENCY: min=0.00, max=9.00, median=5.00
    DURATION: min=4.00, max=95.00, median=14.00
    SEVERITY: min=2.16, max=74.59, median=15.08
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/ssp126/drought_metrics_parana_ssp126_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=0.00, max=10.00, median=4.00
    DURATION: min=2.00, max=120.00, median=14.00
    SEVERITY: min=2.61, max=92.36, median=13.72
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/ssp126/drought_metrics

/tmp/ipykernel_2337151/3510858072.py:176: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_2337151/3510858072.py:177: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_2337151/3510858072.py:178: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/ssp126/drought_metrics_parana_ssp126_ensemble.nc

  Ensemble summary — ssp126:
    FREQUENCY: min=0.00, max=8.00, median=4.00
    DURATION: min=6.50, max=42.50, median=15.50
    SEVERITY: min=4.63, max=32.04, median=15.13

SCENARIO: SSP585 | BASIN: PARANA

  Processing: MPI-ESM1-2-HR
    FREQUENCY: min=0.00, max=9.00, median=4.00
    DURATION: min=3.00, max=130.00, median=14.00
    SEVERITY: min=2.43, max=80.63, median=14.60
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/ssp585/drought_metrics_parana_ssp585_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=0.00, max=9.00, median=4.00
    DURATION: min=2.00, max=88.50, median=16.00
    SEVERITY: min=2.49, max=62.26, median=16.72
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/ssp585/drought_metrics_parana_ssp58

/tmp/ipykernel_2337151/3510858072.py:176: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_2337151/3510858072.py:177: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)


    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/ssp585/drought_metrics_parana_ssp585_ensemble.nc

  Ensemble summary — ssp585:
    FREQUENCY: min=0.00, max=7.00, median=4.00
    DURATION: min=7.00, max=73.00, median=15.25
    SEVERITY: min=6.63, max=43.96, median=14.94

COMPLETED — Rio Parana basin, 5 models, 3 scenarios
Output: /data/brussel/vo/000/bvo00033/vsc11346/SSI_R/5.Caracterizacion/2.Evento/2.Outputs/Rio_Parana/


/tmp/ipykernel_2337151/3510858072.py:178: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)
